In [ ]:
!pip install pandas scikit-learn joblib --quiet

import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

drive_path = "/content/drive/MyDrive/Dmml Project_"
model_path = os.path.join(drive_path, "new_model.pkl")
dataset_path = os.path.join(drive_path, "new_df.csv")

ssm_df = pd.read_csv(os.path.join(drive_path, "Students-Social-Media-Addiction.csv"), sep=';')
vp_df = pd.read_csv(os.path.join(drive_path, "Vehicle-Price-1.csv"))
health_df = pd.read_csv(os.path.join(drive_path, "healthcare_dataset.csv"))
smoke_df = pd.read_csv(os.path.join(drive_path, "smoking_health_data_final.csv"))
mental_df = pd.read_csv(os.path.join(drive_path, "students_mental_health_survey.csv"))

if 'Gender' in ssm_df.columns:
    ssm_df['Gender'] = ssm_df['Gender'].replace({1: 'Female', 2: 'Male', 'Male': 'Male', 'Female': 'Female'})
else:
    ssm_df['Gender'] = 'Unknown'

if 'Gender' not in vp_df.columns:
    vp_df['Gender'] = 'Unknown'

if 'Gender' in health_df.columns:
    health_df['Gender'] = health_df['Gender'].replace({'Male': 'Male', 'Female': 'Female'})
else:
    health_df['Gender'] = 'Unknown'

if 'gender' in smoke_df.columns:
    smoke_df['Gender'] = smoke_df['gender'].replace({'male': 'Male', 'female': 'Female'})
    smoke_df.drop('gender', axis=1, inplace=True)
elif 'Gender' in smoke_df.columns:
    smoke_df['Gender'] = smoke_df['Gender'].replace({'Male': 'Male', 'Female': 'Female'})
else:
    smoke_df['Gender'] = 'Unknown'

if 'Gender' in mental_df.columns:
    mental_df['Gender'] = mental_df['Gender'].replace({'Male': 'Male', 'Female': 'Female'})
else:
    mental_df['Gender'] = 'Unknown'

rename_map = {
    'Health_Risk':'Healthcare_Risk',
    'smoking_status':'Smoking_Risk',
    'Mental_Health_Score':'Mental_Health_Score'
}

if 'Health_Risk' in health_df.columns:
    health_df.rename(columns={'Health_Risk':'Healthcare_Risk'}, inplace=True)

if 'smoking_status' in smoke_df.columns:
    smoke_df.rename(columns={'smoking_status':'Smoking_Risk'}, inplace=True)

if 'Mental_Health_Score' in mental_df.columns:
    mental_df.rename(columns={'Mental_Health_Score':'Mental_Health_Score'}, inplace=True)


if 'age' in ssm_df.columns:
    ssm_df.rename(columns={'age':'age'}, inplace=True)
elif 'Age' in ssm_df.columns:
     ssm_df.rename(columns={'Age':'age'}, inplace=True)
else:
    ssm_df['age'] = 0

if 'Age' in vp_df.columns:
    vp_df.rename(columns={'Age':'age'}, inplace=True)
elif 'age' in vp_df.columns:
     vp_df.rename(columns={'age':'age'}, inplace=True)
else:
    vp_df['age'] = 0


if 'Age' in health_df.columns:
    health_df.rename(columns={'Age':'age'}, inplace=True)
elif 'age' in health_df.columns:
    health_df.rename(columns={'age':'age'}, inplace=True)
else:
    health_df['age'] = 0


if 'age' in smoke_df.columns:
    smoke_df.rename(columns={'age':'age'}, inplace=True)
elif 'Age' in smoke_df.columns:
    smoke_df.rename(columns={'Age':'age'}, inplace=True)
else:
    smoke_df['age'] = 0

if 'Age' in mental_df.columns:
    mental_df.rename(columns={'Age':'age'}, inplace=True)
elif 'age' in mental_df.columns:
    mental_df.rename(columns={'age':'age'}, inplace=True)
else:
    mental_df['age'] = 0


merged_df = ssm_df[['age','Gender']].copy()

if 'Healthcare_Risk' in health_df.columns:
    merged_df = merged_df.merge(health_df[['age','Gender','Healthcare_Risk']], on=['age','Gender'], how='outer')
else:
     merged_df['Healthcare_Risk'] = 0


if 'Smoking_Risk' in smoke_df.columns:
    merged_df = merged_df.merge(smoke_df[['age','Gender','Smoking_Risk']], on=['age','Gender'], how='outer')
else:
    merged_df['Smoking_Risk'] = 0

if 'Mental_Health_Score' in mental_df.columns:
    merged_df = merged_df.merge(mental_df[['age','Gender','Mental_Health_Score']], on=['age','Gender'], how='outer')
else:
    merged_df['Mental_Health_Score'] = 0

merged_df['SSM_Risk'] = 0
merged_df['VP_Risk'] = 0

health_df = merged_df

health_df.fillna(0, inplace=True)
health_df['Healthcare_Risk'] = pd.to_numeric(health_df['Healthcare_Risk'], errors='coerce').fillna(0)
health_df['Smoking_Risk'] = pd.to_numeric(health_df['Smoking_Risk'], errors='coerce').fillna(0)
health_df['Mental_Health_Score'] = pd.to_numeric(health_df['Mental_Health_Score'], errors='coerce').fillna(0)
health_df['SSM_Risk'] = pd.to_numeric(health_df['SSM_Risk'], errors='coerce').fillna(0)
health_df['VP_Risk'] = pd.to_numeric(health_df['VP_Risk'], errors='coerce').fillna(0)

health_df['Health_Risk'] = (health_df[['Healthcare_Risk','Smoking_Risk','Mental_Health_Score','SSM_Risk','VP_Risk']].sum(axis=1) > 2).astype(int)

health_df.to_csv(dataset_path, index=False)
print(f"✅ Merged dataset saved at: {dataset_path}")

X = health_df[['Healthcare_Risk','Smoking_Risk','Mental_Health_Score','SSM_Risk','VP_Risk']]
y = health_df['Health_Risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

joblib.dump(model, model_path)
print(f"✅ Trained model saved at: {model_path}")

print("\nSample merged dataset:")
display(health_df.head())

def get_input(prompt):
    while True:
        try:
            value = float(input(prompt))
            return value
        except ValueError:
            print("Please enter a valid number.")

print("\n=== Interactive New Patient Prediction ===")

while True:
    healthcare_risk = get_input("Enter Healthcare Risk (0-1): ")
    smoking_risk = get_input("Enter Smoking Risk (0-1): ")
    mental_health_score = get_input("Enter Mental Health Score (0-1): ")
    ssm_risk = get_input("Enter SSM Risk (0-1): ")
    vp_risk = get_input("Enter VP Risk (0-1): ")

    new_patient = pd.DataFrame({
        'Healthcare_Risk': [healthcare_risk],
        'Smoking_Risk': [smoking_risk],
        'Mental_Health_Score': [mental_health_score],
        'SSM_Risk': [ssm_risk],
        'VP_Risk': [vp_risk]
    })

    pred = model.predict(new_patient)[0]
    risk_label = "High Risk" if pred==1 else "Low Risk"
    print("\nPredicted Health Risk:", risk_label)

    again = input("\nDo you want to predict another patient? (y/n): ").strip().lower()
    if again != 'y':
        print("✅ Exiting interactive prediction.")
        break

Mounted at /content/drive
✅ Merged dataset saved at: /content/drive/MyDrive/Project/health_df.csv
✅ Trained model saved at: /content/drive/MyDrive/Project/health_model.pkl

Sample merged dataset:

=== Interactive New Patient Prediction ===


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-3471408715.py", line 153, in <cell line: 0>
    cardio_risk = get_input("Enter Cardio Risk (0-1): ")
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-3471408715.py", line 145, in get_input
    value = float(input(prompt))
                  ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelbase.py", line 1177, in raw_input
    return self._input_request(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelbase.py", line 1219, in _input_request
    raise KeyboardInterrupt("Interrupted by user") from None
KeyboardInterrupt: Interrupted by user

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/

TypeError: object of type 'NoneType' has no len()